# Reindexar los embeddings de Aliado Libre en GPU

El índice actual trunca el 76% de los fragmentos a 128 tokens y usa un modelo de paráfrasis donde hace falta uno de recuperación. Esto lo recalcula con `multilingual-e5-large` a 512 tokens.

**Entorno de ejecución → Cambiar tipo de entorno → T4 GPU** antes de empezar.

In [1]:
import threading
import time

def keep_alive():
    print("Iniciando hilo para mantener vivo el entorno...")
    while True:
        print(f"[{time.strftime('%H:%M:%S')}] Colab Keep-Alive: ¡Todavía aquí!")
        time.sleep(300) # Imprime cada 5 minutos

# Iniciar el hilo en segundo plano
keep_alive_thread = threading.Thread(target=keep_alive)
keep_alive_thread.daemon = True # Permite que el programa principal se cierre incluso si el hilo está corriendo
keep_alive_thread.start()

print("Hilo 'keep-alive' iniciado. Busca mensajes cada 5 minutos en la salida.")

Iniciando hilo para mantener vivo el entorno...
[00:39:10] Colab Keep-Alive: ¡Todavía aquí!
Hilo 'keep-alive' iniciado. Busca mensajes cada 5 minutos en la salida.


In [2]:
!pip install -q sentence-transformers huggingface_hub
import torch
assert torch.cuda.is_available(), (
    "No hay GPU. Entorno de ejecucion -> Cambiar tipo de entorno -> T4 GPU."
)
print("GPU:", torch.cuda.get_device_name(0))

GPU: Tesla T4


## Credenciales y parámetros

In [4]:
import getpass, os
# Token de Hugging Face CON permiso de escritura (se pega a mano; no queda en el
# notebook ni en el repositorio).
os.environ["HF_TOKEN"] = getpass.getpass("HF_TOKEN (write): ").strip()

REPO_DATOS = "Gullax/indice-legal-colombia"   # de donde se baja el corpus
REPO_SALIDA = "Gullax/indice-legal-colombia"  # donde se suben los vectores nuevos
MODELO = "intfloat/multilingual-e5-large"     # 1024 dim, 512 tokens, asimetrico
LOTE = 128

HF_TOKEN (write): ··········


## Bajar el corpus ya publicado y extraer los textos

El corpus se baja de Hugging Face, no se sube desde el PC.

In [4]:
from huggingface_hub import hf_hub_download
import sqlite3, json, os

print("Bajando chroma.sqlite3 (~7,9 GB)...")
ruta = hf_hub_download(
    repo_id=REPO_DATOS, filename="chroma.sqlite3",
    repo_type="dataset", token=os.environ["HF_TOKEN"],
)
print("en", ruta)

con = sqlite3.connect(f"file:{ruta}?mode=ro", uri=True)

# El texto y la metadata viven en embedding_metadata (una fila por clave);
# se reconstruye un registro por fragmento. Se lee de una vez y en orden de id
# para que el orden sea reproducible entre corridas.
print("Extrayendo textos...")
campos = {}
for fila_id, clave, valor in con.execute(
    "SELECT id, key, string_value FROM embedding_metadata"
):
    campos.setdefault(fila_id, {})[clave] = valor

ids_por_fila = dict(con.execute("SELECT id, embedding_id FROM embeddings"))

registros = []
for fila_id, datos in campos.items():
    texto = (datos.get("chroma:document") or "").strip()
    frag_id = ids_por_fila.get(fila_id)
    if not texto or not frag_id:
        continue
    registros.append({
        "id": frag_id,
        "texto": texto,
        "documento_id": datos.get("documento_id"),
        "identificador_documento": datos.get("identificador_documento"),
        "titulo_documento": datos.get("titulo_documento"),
        "fuente": datos.get("fuente"),
        "url_original": datos.get("url_original"),
        "orden": datos.get("orden"),
    })

registros.sort(key=lambda r: r["id"])
print(f"{len(registros)} fragmentos listos")
with open("fragmentos.jsonl", "w", encoding="utf-8") as f:
    for r in registros:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

Bajando chroma.sqlite3 (~7,9 GB)...


chroma.sqlite3: reconstructing file:   0%|          |  0.00B / 8.51GB            

chroma.sqlite3: downloading bytes:           |  0.00B            

en /root/.cache/huggingface/hub/datasets--Gullax--indice-legal-colombia/snapshots/a501530a4def82968c49e4f5ef92eabf38c2dd8f/chroma.sqlite3
Extrayendo textos...
718388 fragmentos listos


## Calcular los embeddings (lo que tarda)

Guarda cada 50.000 fragmentos: si Colab corta la sesión, se reanuda.

In [5]:
import numpy as np, json, os, time
from sentence_transformers import SentenceTransformer

modelo = SentenceTransformer(MODELO, device="cuda")
modelo.max_seq_length = 512   # el punto de todo el ejercicio: dejar de truncar
print("max_seq_length:", modelo.max_seq_length,
      "| dim:", modelo.get_sentence_embedding_dimension())

registros = [json.loads(l) for l in open("fragmentos.jsonl", encoding="utf-8")]
# e5 exige prefijos y distingue documento de consulta. Sin esto el modelo rinde
# bastante peor: es literalmente como fue entrenado.
textos = ["passage: " + r["texto"] for r in registros]

# Por trozos, guardando cada uno: una sesion de Colab se puede cortar, y
# perder tres horas de GPU por no guardar seria absurdo.
TROZO = 50_000
os.makedirs("vectores", exist_ok=True)
inicio = time.time()
for i in range(0, len(textos), TROZO):
    salida = f"vectores/parte_{i // TROZO:03d}.npy"
    if os.path.exists(salida):
        print("ya estaba:", salida)
        continue
    vecs = modelo.encode(
        textos[i:i + TROZO], batch_size=LOTE, convert_to_numpy=True,
        normalize_embeddings=True, show_progress_bar=True,
    ).astype("float16")   # la mitad de tamano; la perdida es despreciable
    np.save(salida, vecs)
    hechos = min(i + TROZO, len(textos))
    transcurrido = (time.time() - inicio) / 60
    print(f"{hechos}/{len(textos)} | {transcurrido:.0f} min | "
          f"faltan ~{transcurrido / hechos * (len(textos) - hechos):.0f} min")

partes = sorted(os.listdir("vectores"))
matriz = np.concatenate([np.load(f"vectores/{p}") for p in partes])
np.save("embeddings.f16.npy", matriz)
print("matriz final:", matriz.shape, matriz.dtype)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.24GB            

...

[15:24:10] Colab Keep-Alive: ¡Todavía aquí!
[15:29:10] Colab Keep-Alive: ¡Todavía aquí!
[15:34:10] Colab Keep-Alive: ¡Todavía aquí!
[15:39:10] Colab Keep-Alive: ¡Todavía aquí!
[15:44:10] Colab Keep-Alive: ¡Todavía aquí!
[15:49:10] Colab Keep-Alive: ¡Todavía aquí!
[15:54:10] Colab Keep-Alive: ¡Todavía aquí!
[15:59:10] Colab Keep-Alive: ¡Todavía aquí!
[16:04:10] Colab Keep-Alive: ¡Todavía aquí!
[16:09:10] Colab Keep-Alive: ¡Todavía aquí!
[16:14:10] Colab Keep-Alive: ¡Todavía aquí!
[16:19:10] Colab Keep-Alive: ¡Todavía aquí!
[16:24:10] Colab Keep-Alive: ¡Todavía aquí!
[16:29:10] Colab Keep-Alive: ¡Todavía aquí!
[16:34:10] Colab Keep-Alive: ¡Todavía aquí!
[16:39:10] Colab Keep-Alive: ¡Todavía aquí!
650000/718388 | 953 min | faltan ~100 min


Batches:   0%|          | 0/391 [00:00<?, ?it/s]

[16:44:10] Colab Keep-Alive: ¡Todavía aquí!
[16:49:10] Colab Keep-Alive: ¡Todavía aquí!
[16:54:10] Colab Keep-Alive: ¡Todavía aquí!
[16:59:10] Colab Keep-Alive: ¡Todavía aquí!
[17:04:10] Colab Keep-Alive: ¡Todavía aquí!
[17:09:10] Colab Keep-Alive: ¡Todavía aquí!
[17:14:10] Colab Keep-Alive: ¡Todavía aquí!
[17:19:10] Colab Keep-Alive: ¡Todavía aquí!
[17:24:10] Colab Keep-Alive: ¡Todavía aquí!
[17:29:10] Colab Keep-Alive: ¡Todavía aquí!
[17:34:10] Colab Keep-Alive: ¡Todavía aquí!
[17:39:10] Colab Keep-Alive: ¡Todavía aquí!
[17:44:10] Colab Keep-Alive: ¡Todavía aquí!
[17:49:10] Colab Keep-Alive: ¡Todavía aquí!
700000/718388 | 1028 min | faltan ~27 min


Batches:   0%|          | 0/144 [00:00<?, ?it/s]

[17:54:10] Colab Keep-Alive: ¡Todavía aquí!
[17:59:10] Colab Keep-Alive: ¡Todavía aquí!
[18:04:10] Colab Keep-Alive: ¡Todavía aquí!
[18:09:10] Colab Keep-Alive: ¡Todavía aquí!
[18:14:10] Colab Keep-Alive: ¡Todavía aquí!
718388/718388 | 1050 min | faltan ~0 min
matriz final: (718388, 1024) float16


## Subir los vectores

In [5]:
from huggingface_hub import HfApi
import json, os

api = HfApi(token=os.environ["HF_TOKEN"])
sufijo = MODELO.split("/")[-1]

api.upload_file(
    path_or_fileobj="embeddings.f16.npy",
    path_in_repo=f"embeddings_{sufijo}.f16.npy",
    repo_id=REPO_SALIDA, repo_type="dataset",
    commit_message=f"Embeddings recalculados con {MODELO} (512 tokens, sin truncar)",
)

# Los ids en el MISMO orden que las filas de la matriz: sin esto los vectores
# no se pueden volver a casar con sus fragmentos.
registros = [json.loads(l) for l in open("fragmentos.jsonl", encoding="utf-8")]
with open("ids.json", "w", encoding="utf-8") as f:
    json.dump([r["id"] for r in registros], f)

api.upload_file(
    path_or_fileobj="ids.json", path_in_repo=f"ids_{sufijo}.json",
    repo_id=REPO_SALIDA, repo_type="dataset",
    commit_message="Orden de los ids que corresponde a la matriz de embeddings",
)
print("Listo. Descargar con index/build_index_desde_vectores.py")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /content/embeddings.f16.npy :   1%|          | 7.94MB / 1.47GB            

No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /content/ids.json           :  51%|#####     | 15.9MB / 31.4MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Listo. Descargar con index/build_index_desde_vectores.py


## Prevención de desconexiones por inactividad

Este script ejecutará una tarea en segundo plano que imprimirá un mensaje cada pocos minutos. Esto puede ayudar a que Colab no considere el entorno inactivo y lo mantenga conectado por más tiempo. Ten en cuenta que esto no evitará la desconexión si cierras la pestaña del navegador o pierdes la conexión a internet por completo.

Para detener el hilo 'keep-alive', simplemente detén la ejecución de la celda de Python o reinicia el entorno de ejecución.

In [6]:
from huggingface_hub import HfApi
import numpy as np, json, io

api = HfApi()
info = api.repo_info(REPO_SALIDA, repo_type="dataset", files_metadata=True)
print("Commit actual:", info.sha[:8], "|", info.last_modified)
for f in info.siblings:
    if sufijo in f.rfilename:
        print(f"  {f.rfilename:45s} {f.size/1e6:10.1f} MB")

# Forma real de la matriz, sin descargar 1.47 GB
from huggingface_hub import hf_hub_download
ruta = hf_hub_download(REPO_SALIDA, f"embeddings_{sufijo}.f16.npy", repo_type="dataset")
mm = np.load(ruta, mmap_mode="r")
ids = json.load(open(hf_hub_download(REPO_SALIDA, f"ids_{sufijo}.json", repo_type="dataset")))

print(f"\nmatriz : {mm.shape}  dtype={mm.dtype}")
print(f"ids    : {len(ids)}")
print(f"CASAN  : {mm.shape[0] == len(ids)}")          # <- lo que de verdad importa
print(f"norma media: {np.linalg.norm(mm[:100].astype('f4'), axis=1).mean():.4f}")
print(f"filas nulas: {int((np.abs(mm[:5000]).sum(axis=1) == 0).sum())} de 5000")

Commit actual: a501530a | 2026-09-05 21:27:26+00:00
  embeddings_multilingual-e5-large.f16.npy          1471.3 MB
  ids_multilingual-e5-large.json                      31.4 MB


embeddings_multilingual-e5-large.f16.npy: reconstructing file:   0%|          |  0.00B / 1.47GB            

embeddings_multilingual-e5-large.f16.npy: downloading bytes:           |  0.00B            

ids_multilingual-e5-large.json: reconstructing file:   0%|          |  0.00B / 31.4MB            

ids_multilingual-e5-large.json: downloading bytes:           |  0.00B            


matriz : (718388, 1024)  dtype=float16
ids    : 718388
CASAN  : True
norma media: 1.0000
filas nulas: 0 de 5000


In [7]:
print(f"ids unicos: {len(set(ids))} de {len(ids)}")

ids unicos: 718388 de 718388


In [8]:
!grep -n "np.load\|astype\|float32" index/build_index_desde_vectores.py

grep: index/build_index_desde_vectores.py: No such file or directory
